# 05. Train Popularity Baseline

Computes non-personalized baseline ranks by counting registrations per module and evaluates it on test data.

In [1]:
import os
import numpy as np
import pandas as pd
import json

np.random.seed(42)

processed_dir = os.path.join("..", "..", "datasets", "processed")
train_df = pd.read_csv(os.path.join(processed_dir, "train.csv"))
test_df = pd.read_csv(os.path.join(processed_dir, "test.csv"))
full_df = pd.read_csv(os.path.join(processed_dir, "recommendation_processed.csv"))

num_items = full_df["item_idx"].nunique()
all_items = set(range(num_items))

# Popularity model: count item frequency in train set
pop_scores = train_df["item_idx"].value_counts().to_dict()

def evaluate_popularity(pop_scores, test_df, all_items, topK=10):
    hr_list, ndcg_list = [], []
    for idx, row in test_df.iterrows():
        u = int(row["user_idx"])
        pos_i = int(row["item_idx"])
        
        # sample 99 negatives
        negs = list(all_items - {pos_i})
        if len(negs) > 99:
            sampled_negs = list(np.random.choice(negs, 99, replace=False))
        else:
            sampled_negs = negs
        
        candidates = [pos_i] + sampled_negs
        # score based on popularity score (negative value for ascending argsort)
        scores = np.array([-pop_scores.get(item, 0) for item in candidates])
        
        # get topK (lowest score in pop_scores is highest popularity)
        top_indices = scores.argsort()[:topK] # lowest negative values first
        top_items = [candidates[i] for i in top_indices]
        
        if pos_i in top_items:
            hr_list.append(1.0)
            rank = top_items.index(pos_i)
            ndcg_list.append(1.0 / np.log2(rank + 2))
        else:
            hr_list.append(0.0)
            ndcg_list.append(0.0)
            
    return np.mean(hr_list), np.mean(ndcg_list)

hr_10, ndcg_10 = evaluate_popularity(pop_scores, test_df, all_items, topK=10)
print(f"Popularity HR@10: {hr_10:.4f}, NDCG@10: {ndcg_10:.4f}")

metrics = {
    "popularity_hr_10": hr_10,
    "popularity_ndcg_10": ndcg_10
}

os.makedirs(os.path.join("..", "..", "outputs", "recommendation"), exist_ok=True)
with open(os.path.join("..", "..", "outputs", "recommendation", "pop_metrics.json"), "w") as f:
    json.dump(metrics, f)

print("Popularity baseline training and evaluation complete.")

Popularity HR@10: 1.0000, NDCG@10: 0.5195


Popularity baseline training and evaluation complete.
